## Import the essentials

In [ ]:
import pandas as pd
import random
import json

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset, DataLoader


try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio -q

## Upload dataset and Work with the data

In [ ]:
from datasets import load_dataset

ds = load_dataset("Sairii/mood_dataset")
ds

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/471 [00:00<?, ?B/s]

(…)-00000-of-00001-762df0d1dea723e8.parquet:   0%|          | 0.00/289k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6561 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 6561
    })
})

In [ ]:
ds["train"][10]

{'text': 'Drawn forward by the promise of renewal, a concept some are clearly resisting.',
 'label': 'hopeful'}

In [ ]:
import random

random_indexes = random.sample(range(len(ds["train"])),5)
print(random_indexes)

random_samples = ds["train"][random_indexes]
random_samples

print(f"[INFO] Random samples from dataset:\n")
for text, label in zip(random_samples["text"], random_samples["label"]):
  print(f"Text: {text} | Label: {label}")

[5013, 5482, 6542, 462, 3448]
[INFO] Random samples from dataset:

Text: My resilience fuels my confidence, making me stronger with each stride. | Label: confident
Text: Every part of me feels heavy, as if I'm carrying the weight of the world on my shoulders. | Label: tired
Text: I'm not mad, I'm just preparing my highly detailed list of grievances. 📜😤 | Label: angry
Text: Building my own path to success, without needing a cheering squad. | Label: hopeful
Text: Trying to navigate each day, but it feels like walking through quicksand. | Label: sad


In [ ]:
# Turn dataset into Dataframe
df = pd.DataFrame(ds["train"])
df

,text,label
0,Forward Motion: I'm moving forward with steady...,hopeful
1,"Just trying to draw a full breath, but the air...",sad
2,"Every part of me is screaming for rest, for a ...",tired
3,Wishing for a distraction from myself.,anxious
4,Fear has no place here when self-assurance lea...,confident
...,...,...
6556,Just trying to breathe through this wave of an...,anxious
6557,"Anxiety is an unwelcome visitor, overstaying i...",anxious
6558,"I'm the main event. Always have been, always w...",confident
6559,"Finding Light: Even on cloudy days, I'm findin...",hopeful


In [ ]:
# See the different emotions there are in the dataset
df["label"].unique()

array(['hopeful', 'sad', 'tired', 'anxious', 'confident', 'angry', 'numb',
       'joy'], dtype=object)

In [ ]:
# Count how many examples per emotion
df["label"].value_counts()

,count
label,
sad,869
angry,865
anxious,818
joy,811
hopeful,810
tired,808
numb,792
confident,788


## Build a HuggingFace Model

### Map emotion labels to numerical IDs

In [ ]:
# Get unique values
unique_emotions = df["label"].unique().tolist()
unique_emotions

['hopeful', 'sad', 'tired', 'anxious', 'confident', 'angry', 'numb', 'joy']

In [ ]:
# Map emotion string to integers
from sklearn.preprocessing import LabelEncoder
df["label"] = df["label"].astype(str).str.strip().str.lower()

le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])



id2label = {i:label for i, label in enumerate(le.classes_)}
label2id = {label: i for i, label in id2label.items()}

In [ ]:
id2label

{0: 'angry',
 1: 'anxious',
 2: 'confident',
 3: 'hopeful',
 4: 'joy',
 5: 'numb',
 6: 'sad',
 7: 'tired'}

In [ ]:
label2id

{'angry': 0,
 'anxious': 1,
 'confident': 2,
 'hopeful': 3,
 'joy': 4,
 'numb': 5,
 'sad': 6,
 'tired': 7}

In [ ]:
df.head()

,text,label
0,Forward Motion: I'm moving forward with steady...,3
1,"Just trying to draw a full breath, but the air...",6
2,"Every part of me is screaming for rest, for a ...",7
3,Wishing for a distraction from myself.,1
4,Fear has no place here when self-assurance lea...,2
...,...,...
6556,Just trying to breathe through this wave of an...,1
6557,"Anxiety is an unwelcome visitor, overstaying i...",1
6558,"I'm the main event. Always have been, always w...",2
6559,"Finding Light: Even on cloudy days, I'm findin...",3


### Split the data

In [ ]:
# Convert DataFrame to HF Dataset
from datasets import Dataset
hf_dataset = Dataset.from_pandas(df)

In [ ]:
# Split data into training, val and test
dataset = hf_dataset.train_test_split(test_size = 0.2, seed = 42)


### Tokenize data

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="distilbert/distilbert-base-uncased",
                                          use_fast=True)
tokenizer

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

DistilBertTokenizerFast(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [ ]:
# Test out tokenizer
tokenizer("I am super happy today")

{'input_ids': [101, 1045, 2572, 3565, 3407, 2651, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [ ]:
# Define a function to tokenize text
def tokenize_text(text):
  """
  Tokenizes a given text and returns the tokenized text.
  """
  return tokenizer(text["text"],
                   padding = True,
                   truncation = True)

In [ ]:
# View and example of the function
example_text = {"text":"I want to go to bed right now", "emotion" : "tired"}

tokenize_text(example_text)

{'input_ids': [101, 1045, 2215, 2000, 2175, 2000, 2793, 2157, 2085, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
# Map tokenize text function to the dataset
tokenized_dataset = dataset.map(function = tokenize_text,
                                batched = True,
                                batch_size = 1000)
tokenized_dataset

Map:   0%|          | 0/5248 [00:00<?, ? examples/s]

Map:   0%|          | 0/1313 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 5248
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 1313
    })
})

## Setting up and evaluation metric

In [ ]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")

def compute_accuracy(eval_pred):
  """
  Computes accuracy for multiclass classification.
  """
  predictions, labels = eval_pred
  predictions = np.argmax(predictions, axis = 1)

  return accuracy_metric.compute(predictions = predictions, references = labels)

## Setting up a model for training

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path = "distilbert/distilbert-base-uncased",
    num_labels = int(len(id2label)),
    id2label = id2label,
    label2id = label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


### Create a directory for saving models

In [ ]:
# Create model output directory
from pathlib import Path

# Create models directory
models_dir = Path("models")
models_dir.mkdir(exist_ok = True)

# Create model save name
model_save_name = "mood_classifier_distilbert"

# Create model save path
model_save_dir = Path(models_dir, model_save_name)

model_save_dir

PosixPath('models/mood_classifier_distilbert')

### Setting up training arguments with `TrainingArguments`

In [ ]:
from transformers import TrainingArguments

BATCH_SIZE = 32

# Create training arguments
training_args = TrainingArguments(
    output_dir = model_save_dir,
    learning_rate = 0.0001,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = BATCH_SIZE,
    num_train_epochs = 5,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit = 3,
    use_cpu = False,
    seed = 42,
    load_best_model_at_end = True,
    logging_strategy = "epoch",
    report_to = "none"
)

### Setting up an instance of Trainer

In [ ]:
from transformers import Trainer

# Setup Trainer instance
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_dataset["train"],
    eval_dataset = tokenized_dataset["test"],
    tokenizer = tokenizer,
    compute_metrics = compute_accuracy
)

trainer

/tmp/ipython-input-25-1243652141.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## Train the model using `trainer.train()`

In [ ]:
# Train the model
results = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.624600,0.177167,0.953542
2,0.115700,0.154090,0.962681
3,0.048600,0.101721,0.977913
4,0.022200,0.102124,0.979436
5,0.011500,0.115100,0.977913


In [ ]:
# Save the model
print(f"[INFO] Saving model to {model_save_dir}")
trainer.save_model(output_dir=model_save_dir)

[INFO] Saving model to models/mood_classifier_distilbert


## Making and evaluating predictions on the test data

In [ ]:
# Predictions on test data
predictions_all = trainer.predict(tokenized_dataset["test"])
prediction_values = predictions_all.predictions
predictions_metrics = predictions_all.metrics

print(f"[INFO] Prediction metrics on the test data:")
predictions_metrics

[INFO] Prediction metrics on the test data:


{'test_loss': 0.10172083973884583,
 'test_accuracy': 0.9779131759329779,
 'test_runtime': 2.4663,
 'test_samples_per_second': 532.382,
 'test_steps_per_second': 17.03}

In [ ]:
import torch
from sklearn.metrics import accuracy_score

# 1. Get prediction probabilities (this is optional, could get the same results with step 2 onwards)
pred_probs = torch.softmax(torch.tensor(prediction_values), dim=1)

# 2. Get the predicted labels
pred_labels = torch.argmax(pred_probs, dim=1)

# 3. Get the true labels
true_labels = dataset["test"]["label"]

# 4. Compare predicted labels to true labels to get the test accuracy
test_accuracy = accuracy_score(y_true=true_labels,
                               y_pred=pred_labels)

print(f"[INFO] Test accuracy: {test_accuracy*100}%")

[INFO] Test accuracy: 97.79131759329779%


In [ ]:
# Make a DataFrame of test predictions
test_predictions_df = pd.DataFrame({
    "text": dataset["test"]["text"],
    "true_label": true_labels,
    "pred_label": pred_labels,
    "pred_prob": torch.max(pred_probs, dim=1).values
})

test_predictions_df.head()

,text,true_label,pred_label,pred_prob
0,Heavy Air: Just trying to breathe through the ...,6,6,0.999025
1,My unwavering belief is that things are always...,3,3,0.999274
2,"Pure, unfiltered happiness. No cap, no flex.",4,4,0.998823
3,My heart feels so full it could burst. ❤️‍🔥💥,4,4,0.998212
4,Almost forgot what a smooth day felt like. Tha...,0,0,0.918407


In [ ]:
# Show 10 examples with low prediction probability
test_predictions_df.sort_values("pred_prob", ascending=True).head(10)

,text,true_label,pred_label,pred_prob
237,This is fine. I'm fine. We're all fine.,0,0,0.512794
289,Exhaustion level: expert.,7,0,0.530879
435,"I’m not just ready, I’m destined.",2,3,0.533346
222,Just trying to maintain a semblance of sanity....,0,0,0.553345
1175,"I've got it all together. The pieces, at least.",0,0,0.569133
1268,My calm demeanor is a professional illusion.,0,0,0.591609
405,"Every part of me feels heavy, as if I'm carryi...",7,6,0.594902
111,Tired: a state of being.,7,7,0.625966
5,Everything's great! (Don't ask me to elaborate.),0,4,0.642716
224,Just keeping it together. With super glue.,0,4,0.643594


## Making and inspecting predictions on custom text data

In [ ]:
# Set up device for making predictions

def set_device():
  if torch.cuda.is_available():
    device = torch.device("cuda")
  else:
    device = torch.device("cpu")

  return device

DEVICE = set_device()
DEVICE

device(type='cuda')

In [ ]:
import torch
from transformers import pipeline

BATCH_SIZE = 32

model_path = "models/mood_classifier_distilbert"

# Create an instance of transformers.pipeline
mood_classifier = pipeline(task = "text-classification",
                model = model_path,
                device = DEVICE,
                top_k = 1,
                batch_size = BATCH_SIZE)

mood_classifier

Device set to use cuda


In [ ]:
custom_sentece = "I am super excited right now"
mood_classifier(custom_sentece)

[[{'label': 'joy', 'score': 0.8503211736679077}]]

## Turning our model into a demo

In [ ]:
from huggingface_hub import login

login()


In [ ]:
# Save our model to the HF Hub
model_upload_url = trainer.push_to_hub(
    commit_message = "Sbudo text classifier"
)

print(f"Model successfully uploaded to the Hugging Face Hub with URL: {model_upload_url}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.37k [00:00<?, ?B/s]

Model successfully uploaded to the Hugging Face Hub with URL: https://huggingface.co/Sairii/mood_classifier_distilbert/tree/main/


In [ ]:
import random

affirmations = {
    "sad": {
        "soft": [
            "Your sadness is valid. Let it exist without shame.",
            "It’s okay to fall apart sometimes—you don’t have to hold it together today.",
            "You are still worthy even when you feel broken.",
            "It's okay to feel sad right now. These feelings are temporary.",
            "You are gentle with myself during this time.",
            "Your feelings are valid, and you can  allow yourself to experience them without judgment.",
            "You are worthy of comfort and peace, even when your are sad.",
            "This sadness will pass, and brighter moments are ahead.",
            "You are taking things one breath at a time.",
            "You are strong enough to feel this, and you are resilient.",
            "You are surrounded by quiet support, even if you can't see it right now.",
            "Healing is happening, gently and slowly.",
            "You are allowed to rest and be kind to myself."

        ],
        "sassy": [
            "Cry if you need to, but don’t unpack and live there.",
            "Yes, it hurts. And yes, you’ll rise anyway.",
            "You’re not pathetic—you’re processing. Now hydrate and glow.",
            "Alright, feeling a bit blue, huh? Snap out of it... gently, of course.",
            "Tears? Honey, you're too fabulous for all that watery drama. But seriously, it's cool.",
            "So, the world's got you down? Don't let it win, sweetie. You're tougher than you think.",
            "Yeah, yeah, sad's a vibe. But let's not make it your *only* vibe, okay?",
            "Feeling sorry for yourself? Cute. Now remember you're a queen/king/monarch and this is just a minor plot twist.",
            "Chin up, buttercup. This gloomy cloud is just a temporary accessory, not your whole outfit.",
            "Cry it out if you must, but then let's get back to being the bad*ss you truly are.",
            "Embrace the angst, whatever. Just don't let it steal your sparkle for too long.",
            "This 'sad' thing? It's totally beneath you. But since you're here, feel it, then ditch it.",
            "You're allowed to be a hot mess, but remember, even hot messes eventually clean up and conquer."
        ]
    },
    "angry": {
        "soft": [
            "Your anger is a signal. Listen to what it’s trying to protect.",
            "It’s okay to feel this. Let it move through you, not control you.",
            "You don’t owe anyone your silence when something hurts.",
            "You're feeling this rage, and you're still choosing to be gentle with your own heart.",
            "Damn right you're mad, but you're redirecting this fire to protect your peace, not destroy it.",
            "Your anger is valid, and you're allowing it to flow through you without letting it consume you.",
            "This frustration is a powerful signal, and you're using it to set strong, kind boundaries.",
            "You're furious, and yet you deserve comfort and calm amidst this storm.",
            "You're embracing this fire within, and you're still holding space for your own healing.",
            "It's okay for you to feel this intense anger, and it's also okay for you to give yourself unwavering support.",
            "Your anger is a fierce protector, and you're still nurturing your inner strength.",
            "You're acknowledging this irritation, and you're channeling it into assertive self-care.",
            "You're letting this anger fuel your resolve, while still treating yourself with radical acceptance."
        ],
        "sassy": [
            "You’re not ‘too much’—they’re just not used to boundaries.",
            "Burning bridges? Babe, sometimes it’s called clearing dead weight.",
            "Pop off if you must—but make it poetic and powerful.",
            "Alright, you're feeling a bit blue, huh? Snap out of it... gently, of course.",
            "Tears? Honey, you're too fabulous for all that watery drama. But seriously, it's cool.",
            "So, the world's got you down? Don't let it win, sweetie. You're tougher than you think.",
            "Yeah, yeah, 'sad's a vibe. But let's not make it your *only* vibe, okay?",
            "Feeling sorry for yourself? Cute. Now remember you're a queen/king/monarch and this is just a minor plot twist.",
            "Chin up, buttercup. This gloomy cloud is just a temporary accessory, not your whole outfit.",
            "Cry it out if you must, but then let's get back to being the bad*ss you truly are.",
            "Embrace the angst, whatever. Just don't let it steal your sparkle for too long.",
            "This 'sad' thing? It's totally beneath you. But since you're here, feel it, then ditch it.",
            "You're allowed to be a hot mess, but remember, even hot messes eventually clean up and conquer."
        ]
    },
    "anxious": {
        "soft": [
            "You are safe in this moment. Breathe through it.",
            "This fear will pass. You’ve survived every wave so far.",
            "You don’t need to fix everything right now. Just be here.",
            "It's okay for your mind to race right now. You are safe.",
            "This feeling of anxiety is temporary, and it will pass.",
            "You can breathe through this. Each breath brings you a little more calm.",
            "Your feelings are valid, and you are strong enough to experience them.",
            "You are doing your best, and that is always enough.",
            "Focus on one small thing you can control right now, and let the rest gently fade.",
            "You are worthy of peace, and it is coming to you now.",
            "Your body is trying to protect you, and you can gently reassure it.",
            "You are capable and resilient. You've navigated tough times before.",
            "You are held and supported, even when it doesn't feel like it."
        ],
        "sassy": [
            "Your brain is lying to you again—don't fall for it.",
            "Anxiety? We’ve met. You don’t run the show, babe.",
            "The drama in your head is not reality. Adjust the volume.",
            "Alright, brain, simmer down! We've got this, no need for a five-alarm fire drill.",
            "So, the worry-monster's making a cameo? Cute. Now let's show it the door, eh?",
            "Deep breaths, darling. You're not going to spontaneously combust, promise.",
            "Feeling all fizzy inside? Chill. You're tougher than your nerves give you credit for.",
            "You're totally rocking this whole 'adulting' thing, even with a side of freak-out.",
            "Don't let your thoughts throw a party without your permission. You're the bouncer here.",
            "Worries are just thoughts in fancy, annoying costumes. See 'em, then ignore 'em.",
            "Yeah, your body's on high alert. Tell it to take a coffee break, you've got this.",
            "You've survived worse, trust me. This is just a plot twist, not the grand finale.",
            "So what if you're a mess? Even masterpieces start with a bit of chaos, right?"
        ]
    },
    "joy": {
        "soft": [
            "Soak it in. You deserve this happiness.",
            "Let yourself be fully in the moment—you earned it.",
            "Joy is medicine. Let it heal you today.",
            "You are a magnet for joy and wonderful experiences.",
            "Every day, you discover new reasons to smile and feel grateful.",
            "Happiness flows to you effortlessly, filling every part of your being.",
            "You choose joy in every moment, no matter what comes your way.",
            "Your heart is open, ready to receive and radiate pure happiness.",
            "You find delight in the simple pleasures and beauty all around you.",
            "You are worthy of abundant joy, and it is your natural state.",
            "Laughter comes easily to you, brightening your day and those around you.",
            "You are alive, vibrant, and bursting with positive energy.",
            "Joy multiplies within you as you share your light with the world."
        ],
        "sassy": [
            "Pop off! You’re glowing and they better deal with it.",
            "Stop second-guessing your joy—you freaking earned it.",
            "Main character moment activated. Don’t play small now.",
            "Oh, look who decided to show up: **happiness**! Took you long enough, but welcome!",
            "Life's serving up smiles, and frankly, you're **eating it up** like it's your last meal.",
            "**Good vibes only**, people! You're radiating joy whether they're ready for it or not.",
            "Seriously, your **sparkle** is blinding. Bet you didn't even try that hard.",
            "Who knew being this **delighted** could be so... *you*? Rock it!",
            "Yeah, you're basically a **joy magnet**. Don't act surprised, you earned it.",
            "Turns out, **happiness looks great on you**. Keep it up, buttercup.",
            "Laughter's your new superpower, and you're wielding it like a **pro**.",
            "**Unapologetically gleeful**, that's you. And honestly? It's a look.",
            "Go on, spread that **contagious cheer**. The world could use a dose of your fabulousness."
        ]
    },
    "tired": {
        "soft": [
            "Rest isn’t lazy—it’s necessary.",
            "You don’t have to do it all. You just have to breathe.",
            "Your body and soul are asking for care. Listen.",
            "It's okay to feel tired. You've done enough for today.",
            "Your body is asking for rest, and you are listening.",
            "You are worthy of deep and restorative sleep.",
            "Gentle peace is settling over you as you allow yourself to unwind.",
            "Every breath brings you closer to rest and calm.",
            "It's safe to let go and simply be tired right now.",
            "You are prioritizing your well-being, and that is powerful.",
            "Healing happens when you allow yourself to truly rest.",
            "You are nourished and re-energized with every moment of stillness.",
            "Tomorrow is a new day, and you are preparing for it with gentle rest."
        ],
        "sassy": [
            "If one more person says ‘self-care’ without offering help, throw a crystal.",
            "You’re not a machine—take a damn nap and reclaim your energy.",
            "Tired doesn’t mean weak. It means you’ve been strong for too long.",
            "Alright, body, you're officially on a go-slow. Time for a power-down.",
            "Feeling like a deflated balloon? Good. You've earned this chill-out session.",
            "Seriously, the bed's calling your name. Don't leave it hanging.",
            "Your energy's tapped? Classic. Time to recharge like the superstar you are.",
            "No, you can't run on fumes forever. Get some rest, you magnificent human.",
            "My brain cells are on vacation. See you when I'm less... this.",
            "You're not lazy, you're in **energy-saving mode**. Smart, really.",
            "Operation Recharge is a go! Prepare for maximum horizontal time.",
            "Conquering the world can wait. Your couch, however, cannot.",
            "You're done. D-O-N-E. Now go get that beauty sleep, you earned it."
        ]
    },
    "numb": {
        "soft": [
            "You’re not broken. You’re protecting yourself.",
            "Numbness is not the end—it’s a pause.",
            "Even when you can’t feel it, you are still loved.",
            "It's okay if you're feeling numb right now. This is a pause.",
            "You are safe in this moment of stillness.",
            "Your feelings are still there, waiting for the right time to emerge.",
            "You are gently holding space for whatever wants to surface, or nothing at all.",
            "This quiet state is temporary, and you're moving through it with grace.",
            "You are allowing yourself to simply be, without pressure or expectation.",
            "Healing is happening, even when it feels like nothing is happening.",
            "You are supported, even if you can't feel it right now.",
            "Be kind to yourself in this space of emotional quiet.",
            "You are whole, even in this moment of numbness."
        ],
        "sassy": [
            "If your heart’s on airplane mode, fine—but don’t crash.",
            "You’re still a whole damn person, even when you feel nothing.",
            "Frozen doesn’t mean gone. You’ll thaw. Slowly. And dramatically.",
            "Okay, emotions took a vacation, huh? Fine, you do you.",
            "Feeling a bit... blank? Don't worry, the 'feels' will be back. Eventually.",
            "So, the emotional dial's stuck on zero? Consider it a much-needed break.",
            "You're in neutral. Smart move. Recharge that emotional battery.",
            "No tears, no drama, just... *nothing*. Honestly, kinda peaceful, right?",
            "Your inner thermostat is set to 'off.' Enjoy the quiet while it lasts.",
            "You're not broken, you're just on a really long commercial break.",
            "When your feelings check out, you're just getting a mental detox. Cheers to that!",
            "This emotional flatline? It's just proof you're a master of self-preservation.",
            "Embrace the void, darling. It's less messy this way, for now."
        ]
    },
    "confident": {
        "soft": [
            "You’ve come so far—own your growth.",
            "You don’t need to dim to be loved.",
            "Stand tall. You were made to be seen.",
            "You are capable and strong, in every quiet step you take.",
            "Your inner strength radiates gently, illuminating your path.",
            "You trust your instincts and listen to your wise inner voice.",
            "You are worthy of all your dreams, and you are gracefully moving towards them.",
            "Your confidence is a calm, steady presence within you.",
            "You embrace your authentic self, knowing you are enough.",
            "You move through the world with quiet assurance and grace.",
            "Your presence is a gentle power that inspires those around you.",
            "You believe in yourself, and that belief softly guides your way.",
            "You are secure in who you are, finding peace in your own unique journey."
        ],
        "sassy": [
            "You’re the whole damn meal. Stop acting like a side dish.",
            "Confidence looks good on you—don’t take it off for anyone.",
            "They should be grateful they even get to witness your glow.",
            "Oh, you thought you couldn't? Cute. Watch you **absolutely crush it**.",
            "Walking into the room like you **own the place**? Yeah, you do.",
            "Your confidence isn't a whisper, honey, it's a **full-blown mic drop**.",
            "You're not just capable, you're **unstoppable**. Get used to it.",
            "Yeah, they're looking. Probably wondering how you got so **fabulously self-assured**.",
            "Doubt? Never heard of her. You're too busy **conquering everything**.",
            "You're not just good, you're **goals**. Period.",
            "You're making confidence look easy, and frankly, it's a **masterpiece**.",
            "They say 'fake it 'til you make it.' You're just **making it** look good.",
            "Go on, give 'em that **'I know exactly what I'm doing'** look. Because you do."
        ]
    },
    "hopeful": {
        "soft": [
            "Hope is your quiet rebellion. Keep it alive.",
            "Trust the seeds you’ve planted. Bloom is coming.",
            "You don’t have to know how—it’s okay to just believe.",
            "Even in the quiet, you feel a gentle stirring of possibility.",
            "A soft light of hope guides your way forward, one peaceful step at a time.",
            "You are opening your heart to new beginnings and beautiful potentials.",
            "Trust that good things are gently unfolding for you right now.",
            "Every dawn brings fresh promise and quiet opportunities for joy.",
            "You are cultivating a garden of hope within your spirit.",
            "Miracles are whispering to you, even in the smallest moments.",
            "You believe in the beauty of what is to come, with calm assurance.",
            "Your dreams are tenderly blossoming, nurtured by your quiet belief.",
            "Hope is a gentle breeze, carrying you towards a brighter tomorrow."
        ],
        "sassy": [
            "You’ve survived too much to doubt what’s coming. Own it.",
            "Manifest like a baddie. Hope hard, dream louder.",
            "The universe isn’t done with you yet. Plot twist incoming.",
            "Alright, universe, show me what you got! You're ready for some good stuff.",
            "Feeling that little spark? Yeah, that's your future being awesome.",
            "Things are about to get good, and frankly, you deserve the glow-up.",
            "You're basically a magnet for positive surprises. Get ready to catch 'em!",
            "Still got that fire in your belly? Good. It's leading you somewhere epic.",
            "Don't just hope for it, expect it. You're manifesting greatness, darling.",
            "Your future's looking so bright, you might need shades. Just sayin'.",
            "Time to turn up the optimism. Your vibe is attracting your tribe... and cool opportunities.",
            "Plot twist: everything's about to work out. You saw it here first.",
            "Yeah, you're optimistic. What about it? It's a good look on you."
        ]
    }
}


In [ ]:
from typing import Dict

classifier = pipeline(task = "text-classification",
              model = model_path,
              device = DEVICE,
              top_k = 2,
              batch_size = BATCH_SIZE)

# Create function to take a string input
def mood_text_classifier(text:str, tone: str = "soft") -> Dict[str,float]:

  # Outputs from the pipeline
  outputs = classifier(text)[0]
  result = outputs[0]

  label = result["label"].lower()
  score = result["score"]

  # Get affirmation
  if label in affirmations:
    affirmation = random.choice(affirmations[label][tone])
  else:
    affirmation = "No affirmation found for this emotion :("

  return{
      "emotion": label,
      "confidence":round(score, 3),
      "affirmation": affirmation
  }

mood_text_classifier(text = "You are building a demo!")

Device set to use cuda


{'emotion': 'angry',
 'confidence': 0.973,
 'affirmation': 'You don’t owe anyone your silence when something hurts.'}

### Build a small Gradio demo

In [ ]:
import gradio as gr

def mood_interface(text, tone):
    result = mood_text_classifier(text, tone = tone)  # you can later add dropdown to choose tone
    return f"🧠 Emotion: {result['emotion'].capitalize()}\n" \
           f"📈 Confidence: {round(result['confidence']*100, 2)}%\n\n" \
           f"💬 Affirmation:\n{result['affirmation']}"

demo = gr.Interface(
    fn=mood_interface,
    inputs=[
        gr.Textbox(lines=4, placeholder="Write your journal entry here...", label="How are you feeling today?"),
        gr.Radio(["soft", "sassy"], label="Affirmation Style", value="soft")
    ],
    outputs=gr.Textbox(label="Mood + Affirmation"),
    title="🧠 Mood Classifier with Affirmations",
    description="Journal your heart out. Choose how you want to be loved today: soft hugs or sassy truth bombs."
)

demo.launch(share = True, debug = True)



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4e6077a2c6bf2fc065.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4e6077a2c6bf2fc065.gradio.live


## Deploy our app

### Making a directory to stores our demo

In [ ]:
from pathlib import Path

# Make directory for demos
demos_dir = Path("../demos")
demos_dir.mkdir(exist_ok = True)

# Create a folder for the classifier demo
mood_classifier_dir = Path(demos_dir, "mood_text_classifier")
mood_classifier_dir.mkdir(exist_ok = True)

### Making an app.py file

In [ ]:
%%writefile ../demos/mood_text_classifier/app.py
# Import require files
import torch
import gradio as gr
import random

from typing import Dict
from transformers import pipeline

# Define model path
model_path = "Sairii/mood_classifier_distilbert"

# Define affirmations
affirmations = {
    "sad": {
        "soft": [
            "Your sadness is valid. Let it exist without shame.",
            "It’s okay to fall apart sometimes—you don’t have to hold it together today.",
            "You are still worthy even when you feel broken.",
            "It's okay to feel sad right now. These feelings are temporary.",
            "You are gentle with myself during this time.",
            "Your feelings are valid, and you can  allow yourself to experience them without judgment.",
            "You are worthy of comfort and peace, even when your are sad.",
            "This sadness will pass, and brighter moments are ahead.",
            "You are taking things one breath at a time.",
            "You are strong enough to feel this, and you are resilient.",
            "You are surrounded by quiet support, even if you can't see it right now.",
            "Healing is happening, gently and slowly.",
            "You are allowed to rest and be kind to myself."

        ],
        "sassy": [
            "Cry if you need to, but don’t unpack and live there.",
            "Yes, it hurts. And yes, you’ll rise anyway.",
            "You’re not pathetic—you’re processing. Now hydrate and glow.",
            "Alright, feeling a bit blue, huh? Snap out of it... gently, of course.",
            "Tears? Honey, you're too fabulous for all that watery drama. But seriously, it's cool.",
            "So, the world's got you down? Don't let it win, sweetie. You're tougher than you think.",
            "Yeah, yeah, sad's a vibe. But let's not make it your *only* vibe, okay?",
            "Feeling sorry for yourself? Cute. Now remember you're a queen/king/monarch and this is just a minor plot twist.",
            "Chin up, buttercup. This gloomy cloud is just a temporary accessory, not your whole outfit.",
            "Cry it out if you must, but then let's get back to being the bad*ss you truly are.",
            "Embrace the angst, whatever. Just don't let it steal your sparkle for too long.",
            "This 'sad' thing? It's totally beneath you. But since you're here, feel it, then ditch it.",
            "You're allowed to be a hot mess, but remember, even hot messes eventually clean up and conquer."
        ]
    },
    "angry": {
        "soft": [
            "Your anger is a signal. Listen to what it’s trying to protect.",
            "It’s okay to feel this. Let it move through you, not control you.",
            "You don’t owe anyone your silence when something hurts.",
            "You're feeling this rage, and you're still choosing to be gentle with your own heart.",
            "Damn right you're mad, but you're redirecting this fire to protect your peace, not destroy it.",
            "Your anger is valid, and you're allowing it to flow through you without letting it consume you.",
            "This frustration is a powerful signal, and you're using it to set strong, kind boundaries.",
            "You're furious, and yet you deserve comfort and calm amidst this storm.",
            "You're embracing this fire within, and you're still holding space for your own healing.",
            "It's okay for you to feel this intense anger, and it's also okay for you to give yourself unwavering support.",
            "Your anger is a fierce protector, and you're still nurturing your inner strength.",
            "You're acknowledging this irritation, and you're channeling it into assertive self-care.",
            "You're letting this anger fuel your resolve, while still treating yourself with radical acceptance."
        ],
        "sassy": [
            "You’re not ‘too much’—they’re just not used to boundaries.",
            "Burning bridges? Babe, sometimes it’s called clearing dead weight.",
            "Pop off if you must—but make it poetic and powerful.",
            "Alright, you're feeling a bit blue, huh? Snap out of it... gently, of course.",
            "Tears? Honey, you're too fabulous for all that watery drama. But seriously, it's cool.",
            "So, the world's got you down? Don't let it win, sweetie. You're tougher than you think.",
            "Yeah, yeah, 'sad's a vibe. But let's not make it your *only* vibe, okay?",
            "Feeling sorry for yourself? Cute. Now remember you're a queen/king/monarch and this is just a minor plot twist.",
            "Chin up, buttercup. This gloomy cloud is just a temporary accessory, not your whole outfit.",
            "Cry it out if you must, but then let's get back to being the bad*ss you truly are.",
            "Embrace the angst, whatever. Just don't let it steal your sparkle for too long.",
            "This 'sad' thing? It's totally beneath you. But since you're here, feel it, then ditch it.",
            "You're allowed to be a hot mess, but remember, even hot messes eventually clean up and conquer."
        ]
    },
    "anxious": {
        "soft": [
            "You are safe in this moment. Breathe through it.",
            "This fear will pass. You’ve survived every wave so far.",
            "You don’t need to fix everything right now. Just be here.",
            "It's okay for your mind to race right now. You are safe.",
            "This feeling of anxiety is temporary, and it will pass.",
            "You can breathe through this. Each breath brings you a little more calm.",
            "Your feelings are valid, and you are strong enough to experience them.",
            "You are doing your best, and that is always enough.",
            "Focus on one small thing you can control right now, and let the rest gently fade.",
            "You are worthy of peace, and it is coming to you now.",
            "Your body is trying to protect you, and you can gently reassure it.",
            "You are capable and resilient. You've navigated tough times before.",
            "You are held and supported, even when it doesn't feel like it."
        ],
        "sassy": [
            "Your brain is lying to you again—don't fall for it.",
            "Anxiety? We’ve met. You don’t run the show, babe.",
            "The drama in your head is not reality. Adjust the volume.",
            "Alright, brain, simmer down! We've got this, no need for a five-alarm fire drill.",
            "So, the worry-monster's making a cameo? Cute. Now let's show it the door, eh?",
            "Deep breaths, darling. You're not going to spontaneously combust, promise.",
            "Feeling all fizzy inside? Chill. You're tougher than your nerves give you credit for.",
            "You're totally rocking this whole 'adulting' thing, even with a side of freak-out.",
            "Don't let your thoughts throw a party without your permission. You're the bouncer here.",
            "Worries are just thoughts in fancy, annoying costumes. See 'em, then ignore 'em.",
            "Yeah, your body's on high alert. Tell it to take a coffee break, you've got this.",
            "You've survived worse, trust me. This is just a plot twist, not the grand finale.",
            "So what if you're a mess? Even masterpieces start with a bit of chaos, right?"
        ]
    },
    "joy": {
        "soft": [
            "Soak it in. You deserve this happiness.",
            "Let yourself be fully in the moment—you earned it.",
            "Joy is medicine. Let it heal you today.",
            "You are a magnet for joy and wonderful experiences.",
            "Every day, you discover new reasons to smile and feel grateful.",
            "Happiness flows to you effortlessly, filling every part of your being.",
            "You choose joy in every moment, no matter what comes your way.",
            "Your heart is open, ready to receive and radiate pure happiness.",
            "You find delight in the simple pleasures and beauty all around you.",
            "You are worthy of abundant joy, and it is your natural state.",
            "Laughter comes easily to you, brightening your day and those around you.",
            "You are alive, vibrant, and bursting with positive energy.",
            "Joy multiplies within you as you share your light with the world."
        ],
        "sassy": [
            "Pop off! You’re glowing and they better deal with it.",
            "Stop second-guessing your joy—you freaking earned it.",
            "Main character moment activated. Don’t play small now.",
            "Oh, look who decided to show up: **happiness**! Took you long enough, but welcome!",
            "Life's serving up smiles, and frankly, you're **eating it up** like it's your last meal.",
            "**Good vibes only**, people! You're radiating joy whether they're ready for it or not.",
            "Seriously, your **sparkle** is blinding. Bet you didn't even try that hard.",
            "Who knew being this **delighted** could be so... *you*? Rock it!",
            "Yeah, you're basically a **joy magnet**. Don't act surprised, you earned it.",
            "Turns out, **happiness looks great on you**. Keep it up, buttercup.",
            "Laughter's your new superpower, and you're wielding it like a **pro**.",
            "**Unapologetically gleeful**, that's you. And honestly? It's a look.",
            "Go on, spread that **contagious cheer**. The world could use a dose of your fabulousness."
        ]
    },
    "tired": {
        "soft": [
            "Rest isn’t lazy—it’s necessary.",
            "You don’t have to do it all. You just have to breathe.",
            "Your body and soul are asking for care. Listen.",
            "It's okay to feel tired. You've done enough for today.",
            "Your body is asking for rest, and you are listening.",
            "You are worthy of deep and restorative sleep.",
            "Gentle peace is settling over you as you allow yourself to unwind.",
            "Every breath brings you closer to rest and calm.",
            "It's safe to let go and simply be tired right now.",
            "You are prioritizing your well-being, and that is powerful.",
            "Healing happens when you allow yourself to truly rest.",
            "You are nourished and re-energized with every moment of stillness.",
            "Tomorrow is a new day, and you are preparing for it with gentle rest."
        ],
        "sassy": [
            "If one more person says ‘self-care’ without offering help, throw a crystal.",
            "You’re not a machine—take a damn nap and reclaim your energy.",
            "Tired doesn’t mean weak. It means you’ve been strong for too long.",
            "Alright, body, you're officially on a go-slow. Time for a power-down.",
            "Feeling like a deflated balloon? Good. You've earned this chill-out session.",
            "Seriously, the bed's calling your name. Don't leave it hanging.",
            "Your energy's tapped? Classic. Time to recharge like the superstar you are.",
            "No, you can't run on fumes forever. Get some rest, you magnificent human.",
            "My brain cells are on vacation. See you when I'm less... this.",
            "You're not lazy, you're in **energy-saving mode**. Smart, really.",
            "Operation Recharge is a go! Prepare for maximum horizontal time.",
            "Conquering the world can wait. Your couch, however, cannot.",
            "You're done. D-O-N-E. Now go get that beauty sleep, you earned it."
        ]
    },
    "numb": {
        "soft": [
            "You’re not broken. You’re protecting yourself.",
            "Numbness is not the end—it’s a pause.",
            "Even when you can’t feel it, you are still loved.",
            "It's okay if you're feeling numb right now. This is a pause.",
            "You are safe in this moment of stillness.",
            "Your feelings are still there, waiting for the right time to emerge.",
            "You are gently holding space for whatever wants to surface, or nothing at all.",
            "This quiet state is temporary, and you're moving through it with grace.",
            "You are allowing yourself to simply be, without pressure or expectation.",
            "Healing is happening, even when it feels like nothing is happening.",
            "You are supported, even if you can't feel it right now.",
            "Be kind to yourself in this space of emotional quiet.",
            "You are whole, even in this moment of numbness."
        ],
        "sassy": [
            "If your heart’s on airplane mode, fine—but don’t crash.",
            "You’re still a whole damn person, even when you feel nothing.",
            "Frozen doesn’t mean gone. You’ll thaw. Slowly. And dramatically.",
            "Okay, emotions took a vacation, huh? Fine, you do you.",
            "Feeling a bit... blank? Don't worry, the 'feels' will be back. Eventually.",
            "So, the emotional dial's stuck on zero? Consider it a much-needed break.",
            "You're in neutral. Smart move. Recharge that emotional battery.",
            "No tears, no drama, just... *nothing*. Honestly, kinda peaceful, right?",
            "Your inner thermostat is set to 'off.' Enjoy the quiet while it lasts.",
            "You're not broken, you're just on a really long commercial break.",
            "When your feelings check out, you're just getting a mental detox. Cheers to that!",
            "This emotional flatline? It's just proof you're a master of self-preservation.",
            "Embrace the void, darling. It's less messy this way, for now."
        ]
    },
    "confident": {
        "soft": [
            "You’ve come so far—own your growth.",
            "You don’t need to dim to be loved.",
            "Stand tall. You were made to be seen.",
            "You are capable and strong, in every quiet step you take.",
            "Your inner strength radiates gently, illuminating your path.",
            "You trust your instincts and listen to your wise inner voice.",
            "You are worthy of all your dreams, and you are gracefully moving towards them.",
            "Your confidence is a calm, steady presence within you.",
            "You embrace your authentic self, knowing you are enough.",
            "You move through the world with quiet assurance and grace.",
            "Your presence is a gentle power that inspires those around you.",
            "You believe in yourself, and that belief softly guides your way.",
            "You are secure in who you are, finding peace in your own unique journey."
        ],
        "sassy": [
            "You’re the whole damn meal. Stop acting like a side dish.",
            "Confidence looks good on you—don’t take it off for anyone.",
            "They should be grateful they even get to witness your glow.",
            "Oh, you thought you couldn't? Cute. Watch you **absolutely crush it**.",
            "Walking into the room like you **own the place**? Yeah, you do.",
            "Your confidence isn't a whisper, honey, it's a **full-blown mic drop**.",
            "You're not just capable, you're **unstoppable**. Get used to it.",
            "Yeah, they're looking. Probably wondering how you got so **fabulously self-assured**.",
            "Doubt? Never heard of her. You're too busy **conquering everything**.",
            "You're not just good, you're **goals**. Period.",
            "You're making confidence look easy, and frankly, it's a **masterpiece**.",
            "They say 'fake it 'til you make it.' You're just **making it** look good.",
            "Go on, give 'em that **'I know exactly what I'm doing'** look. Because you do."
        ]
    },
    "hopeful": {
        "soft": [
            "Hope is your quiet rebellion. Keep it alive.",
            "Trust the seeds you’ve planted. Bloom is coming.",
            "You don’t have to know how—it’s okay to just believe.",
            "Even in the quiet, you feel a gentle stirring of possibility.",
            "A soft light of hope guides your way forward, one peaceful step at a time.",
            "You are opening your heart to new beginnings and beautiful potentials.",
            "Trust that good things are gently unfolding for you right now.",
            "Every dawn brings fresh promise and quiet opportunities for joy.",
            "You are cultivating a garden of hope within your spirit.",
            "Miracles are whispering to you, even in the smallest moments.",
            "You believe in the beauty of what is to come, with calm assurance.",
            "Your dreams are tenderly blossoming, nurtured by your quiet belief.",
            "Hope is a gentle breeze, carrying you towards a brighter tomorrow."
        ],
        "sassy": [
            "You’ve survived too much to doubt what’s coming. Own it.",
            "Manifest like a baddie. Hope hard, dream louder.",
            "The universe isn’t done with you yet. Plot twist incoming.",
            "Alright, universe, show me what you got! You're ready for some good stuff.",
            "Feeling that little spark? Yeah, that's your future being awesome.",
            "Things are about to get good, and frankly, you deserve the glow-up.",
            "You're basically a magnet for positive surprises. Get ready to catch 'em!",
            "Still got that fire in your belly? Good. It's leading you somewhere epic.",
            "Don't just hope for it, expect it. You're manifesting greatness, darling.",
            "Your future's looking so bright, you might need shades. Just sayin'.",
            "Time to turn up the optimism. Your vibe is attracting your tribe... and cool opportunities.",
            "Plot twist: everything's about to work out. You saw it here first.",
            "Yeah, you're optimistic. What about it? It's a good look on you."
        ]
    }
}


classifier = pipeline(task = "text-classification",
              model = model_path,
              device = "cuda" if torch.cuda.is_available() else "cpu",
              top_k = 2,
              batch_size = 32)


# Define our function to use with our model
def mood_text_classifier(text:str, tone: str = "soft") -> Dict[str,float]:

  # Outputs from the pipeline
  outputs = classifier(text)[0]
  result = outputs[0]

  label = result["label"].lower()
  score = result["score"]

  # Get affirmation
  if label in affirmations:
    affirmation = random.choice(affirmations[label][tone])
  else:
    affirmation = "No affirmation found for this emotion :("

  return{
      "emotion": label,
      "confidence":round(score, 3),
      "affirmation": affirmation
  }

# Create the gradio interface
def mood_interface(text, tone):
    result = mood_text_classifier(text, tone = tone)  # you can later add dropdown to choose tone
    return f"🧠 Emotion: {result['emotion'].capitalize()}\n" \
           f"📈 Confidence: {round(result['confidence']*100, 2)}%\n\n" \
           f"💬 Affirmation:\n{result['affirmation']}"

demo = gr.Interface(
    fn=mood_interface,
    inputs=[
        gr.Textbox(lines=4, placeholder="Write your journal entry here...", label="How are you feeling today?"),
        gr.Radio(["soft", "sassy"], label="Affirmation Style", value="soft")
    ],
    outputs=gr.Textbox(label="Mood + Affirmation"),
    title="🧠 Mood Classifier with Affirmations",
    description="Journal your heart out. Choose how you want to be loved today: soft hugs or sassy truth bombs."
)

# Launch the interface
if __name__ == "__main__":
  demo.launch()

Overwriting ../demos/mood_text_classifier/app.py


### Making a README

In [ ]:
%%writefile ../demos/mood_text_classifier/README.md

---
title : Mood Text Classifier
emoji: ❤️🤬😪😴
colorFrom: blue
colorTo: yellow
sdk: gradio
app_file: app.py
pinned: false
licence: apache-2.0
---

# ❤️🤬😪😴 Mood Text Classifier
Small mood classifier where it determines the mood about your text.
DistilBERT model with a database of [6561 mood captions](https://huggingface.co/datasets/Sairii/mood_dataset?library=datasets)


Overwriting ../demos/mood_text_classifier/README.md


### Making a requiments file

In [ ]:
%%writefile ../demos/mood_text_classifier/requirements.txt
gradio
torch
transformers

Overwriting ../demos/mood_text_classifier/requirements.txt


### Uploading to HuggingFace Spaces

In [ ]:
# 1. Import the required methods for uploading to the HF Hub
from huggingface_hub import(
    create_repo,
    get_full_repo_name,
    upload_file,
    upload_folder
)

# 2. Define the parameters we'd like to use for uploading our Space
LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD ="../demos/mood_text_classifier"
HF_TARGET_SPACE_NAME = "mood_text_classifier"
HF_REPO_TYPE = "space"
HF_SPACE_SDK="gradio"

# 3. Create a Space repo on Hugging Face Hub
print(f"[INFO] Creating repo on Hugging Face Hub with name: {HF_TARGET_SPACE_NAME}")
create_repo(
    repo_id=HF_TARGET_SPACE_NAME,
    repo_type = HF_REPO_TYPE,
    private=False,
    space_sdk = HF_SPACE_SDK,
    exist_ok = True

)

# 4. Get the full repo name(e.g. {username}/{repo_name})
hf_full_repo_name = get_full_repo_name(model_id = HF_TARGET_SPACE_NAME)
print(f"[INFO] Full Hugging Face Hub repo name: {hf_full_repo_name}")

# 5.Upload our demo folder
print(f"[INFO] Uploading {LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD} to repo:{hf_full_repo_name}")
folder_upload_url = upload_folder(
    repo_id = hf_full_repo_name,
    folder_path = LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD,
    path_in_repo = ".",
    repo_type=HF_REPO_TYPE,
    commit_message = "Uploading our mood classifier demos from  a notebook"

)

print(f"[INFO] Demo folder successfully with commit URL: {folder_upload_url} ")

[INFO] Creating repo on Hugging Face Hub with name: mood_text_classifier
[INFO] Full Hugging Face Hub repo name: Sairii/mood_text_classifier
[INFO] Uploading ../demos/mood_text_classifier to repo:Sairii/mood_text_classifier
[INFO] Demo folder successfully with commit URL: https://huggingface.co/spaces/Sairii/mood_text_classifier/tree/main/. 
